# 02 · Team Engagement & Themes
Team-wide view of customer engagement, coverage breadth, and AI-extracted conversation themes from Gong summaries.
Adjust the date range in the filters cell.

In [ ]:
import datetime, json
import matplotlib.pyplot as plt, matplotlib.patches as mpatches, matplotlib.gridspec as gs
import pandas as pd
from IPython.display import display, HTML
plt.switch_backend("agg")

SF_BLUE="#29B5E8"; SF_TEAL="#00A9CE"; SF_GREEN="#36B37E"
SF_ORANGE="#FF8B00"; SF_DARK="#0A2859"; SF_GRAY="#D0D0D0"; BG="#F9FAFB"

def clean(ax):
    ax.set_facecolor(BG)
    for s in ax.spines.values(): s.set_visible(False)

def fmt_eacv(v):
    if not v or (isinstance(v,float) and str(v)=="nan"): return "$0"
    v=float(v)
    return f"${v/1e6:.1f}M" if v>=1e6 else f"${v/1e3:.0f}K" if v>=1e3 else f"${v:.0f}"

def html_table(df, row_color_fn=None):
    if hasattr(df, 'to_pandas'): df = df.to_pandas()
    html = "<table style='border-collapse:collapse;width:100%;font-size:13px;font-family:sans-serif'>"
    html += "<tr style='background:#0A2859;color:white'>" + "".join("<th style='padding:8px 12px;text-align:left'>" + str(col) + "</th>" for col in df.columns) + "</tr>"
    for i, (_, r) in enumerate(df.iterrows()):
        bg = row_color_fn(r) if row_color_fn else ("#f8f9fa" if i%2==0 else "white")
        html += "<tr style='background:" + bg + "'>" + "".join("<td style='padding:7px 12px'>" + str(v if v is not None else '') + "</td>" for v in r) + "</tr>"
    html += "</table>"
    display(HTML(html))

TEAM = {
    "Jim Lebonitte": "005VI00000Z0y6bYAB",
    "Michael Hamilton": "005VI00000ExC3VYAV",
    "Patrick Sheehan": "0053r00000BblkZAAR",
    "Phani Alapaty": "005VI00000HajknYAB",
    "Steve Mitchener": "0050Z000009IrKRQA0",
    "Zaki Bajwa": "005VI00000XibQfYAJ"
}
TEAM_IDS   = list(TEAM.values())
team_ids_str = "', '".join(TEAM_IDS)
ACT_TABLE  = "TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES"

fy_start   = datetime.date(2026, 2, 1)
q2_start   = datetime.date(2026, 5, 1)
today      = datetime.date.today()
fy_start_str = str(fy_start)
q2_start_str = str(q2_start)
today_str    = str(today)
print("Setup complete ✓")

# ─── FILTERS ─────────────────────────────────────────────
period_start = datetime.date(2026, 4, 1)
period_end   = today
# ─────────────────────────────────────────────────────────
ps, pe = str(period_start), str(period_end)
print(f"Team view  |  {ps} → {pe}")


## Team Activity Summary

In [ ]:
%%sql -r team_summary
SELECT
    COALESCE(MEETING_SE_NAME, SOURCE_FILE)          AS specialist,
    COUNT(*)                                         AS total_meetings,
    COUNT(DISTINCT SF_ACCOUNT_ID)                    AS unique_accounts,
    SUM(IFF(SF_ACTIVITY_ID IS NOT NULL,1,0))         AS setsail_matched,
    SUM(IFF(USE_CASE_TAGGED_IN_SF='Yes',1,0))        AS with_use_case,
    SUM(IFF(USE_CASE_TAGGED_IN_SF='No' AND SF_ACCOUNT_ID IS NOT NULL,1,0)) AS uc_gap,
    SUM(IFF(SUMMARY IS NOT NULL,1,0))                AS with_gong_summary,
    ROUND(100.0*SUM(IFF(SF_ACTIVITY_ID IS NOT NULL,1,0))/COUNT(*),1) AS setsail_match_pct
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_DATE BETWEEN '{{ps}}' AND '{{pe}}'
GROUP BY 1 ORDER BY 2 DESC

In [ ]:
team_summary = team_summary.to_pandas() if hasattr(team_summary, "to_pandas") else team_summary
html_table(team_summary)

## Weekly Engagement — All Specialists

In [ ]:
%%sql -r weekly_team
SELECT
    DATE_TRUNC('week', MEETING_DATE)::DATE AS week,
    COALESCE(MEETING_SE_NAME, 'Unknown')   AS specialist,
    COUNT(*)                               AS meetings
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_DATE BETWEEN '{{ps}}' AND '{{pe}}'
GROUP BY 1,2 ORDER BY 1,2

In [ ]:
weekly_team = weekly_team.to_pandas() if hasattr(weekly_team, "to_pandas") else weekly_team
df = weekly_team
pivot = df.pivot(index="WEEK", columns="SPECIALIST", values="MEETINGS").fillna(0)
fig, ax = plt.subplots(figsize=(14, 4)); clean(ax)
bottom = None
colors = [SF_BLUE, SF_TEAL, SF_GREEN, SF_ORANGE, "#9B59B6", SF_DARK]
for i, col in enumerate(pivot.columns):
    ax.bar(pivot.index.astype(str), pivot[col], bottom=bottom,
           label=col, color=colors[i % len(colors)], alpha=0.85)
    bottom = pivot[col] if bottom is None else bottom + pivot[col]
ax.set_title("Team Customer Meetings by Week", fontsize=13, color=SF_DARK)
ax.set_xlabel("Week"); ax.set_ylabel("Meetings")
ax.legend(loc="upper left", frameon=False, fontsize=9)
plt.xticks(rotation=40, ha="right"); plt.tight_layout(); plt.show()

## Top Accounts — Team Coverage

In [ ]:
%%sql -r top_team_accts
SELECT
    COALESCE(CUSTOMER,'(unknown)')   AS customer,
    MAX(SF_ACCOUNT_ID)               AS sf_account_id,
    COUNT(*)                         AS total_meetings,
    COUNT(DISTINCT MEETING_SE_NAME)  AS specialists_engaged,
    MAX(MEETING_DATE)                AS last_touch,
    DATEDIFF('day',MAX(MEETING_DATE),CURRENT_DATE()) AS days_since,
    MAX(USE_CASE_TAGGED_IN_SF)       AS uc_status,
    MAX(USE_CASE_NAME)               AS use_case
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_DATE BETWEEN '{{ps}}' AND '{{pe}}'
  AND CUSTOMER IS NOT NULL AND CUSTOMER != ''
GROUP BY 1 ORDER BY 3 DESC LIMIT 30

In [ ]:
top_team_accts = top_team_accts.to_pandas() if hasattr(top_team_accts, "to_pandas") else top_team_accts
def row_color(r):
    d = int(r.get("DAYS_SINCE") or 0)
    s = str(r.get("UC_STATUS",""))
    if s=="Yes": return "#d4edda"
    if d>30: return "#fdecea"
    if s=="No": return "#fff3cd"
    return "white"
html_table(top_team_accts, row_color_fn=row_color)

## AI — Key Themes Across All Customer Conversations

In [ ]:
%%sql -r all_summaries
SELECT MEETING_DATE, CUSTOMER, MEETING_SE_NAME, SUMMARY
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_DATE BETWEEN '{{ps}}' AND '{{pe}}'
  AND SUMMARY IS NOT NULL AND TRIM(SUMMARY) != ''
ORDER BY MEETING_DATE DESC LIMIT 60

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
all_summaries = all_summaries.to_pandas() if hasattr(all_summaries, "to_pandas") else all_summaries
texts = "\n---\n".join(
    f"[{r['MEETING_SE_NAME']} @ {r['CUSTOMER']} on {r['MEETING_DATE']}] {r['SUMMARY']}"
    for _, r in all_summaries.iterrows() if r['SUMMARY']
)
if texts.strip():
    prompt = (
        "You are analyzing Gong meeting summaries for a Snowflake specialist team.\n"
        "Identify the top 7 themes and patterns you see across ALL customer conversations.\n"
        "For each theme: theme name, which customers mentioned it, frequency, and a 1-sentence insight.\n"
        "Also identify: any competitor mentions, product/feature gaps customers are raising, "
        "and the top 3 strategic opportunities for the team to pursue.\n\nSummaries:\n" + texts[:6000]
    )
    result = session.sql("SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-7b', ?)", [prompt]).collect()[0][0]
    print(f"=== Team Conversation Themes ({ps} → {pe}) ===\n")
    print(result)
else:
    print("No Gong summaries available — run weekly reports with --gsheet first.")

## Coverage Gaps — Accounts Going Cold

In [ ]:
%%sql -r cold_accounts
SELECT
    COALESCE(CUSTOMER,'(unknown)')  AS customer,
    MAX(SF_ACCOUNT_ID)              AS sf_account_id,
    MAX(MEETING_SE_NAME)            AS last_specialist,
    MAX(MEETING_DATE)               AS last_touch,
    DATEDIFF('day',MAX(MEETING_DATE),CURRENT_DATE()) AS days_since_contact,
    MAX(USE_CASE_TAGGED_IN_SF)      AS uc_status,
    MAX(USE_CASE_NAME)              AS use_case
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE CUSTOMER IS NOT NULL AND CUSTOMER != ''
GROUP BY 1
HAVING DATEDIFF('day',MAX(MEETING_DATE),CURRENT_DATE()) > 30
ORDER BY 5 DESC LIMIT 20

In [ ]:
cold_accounts = cold_accounts.to_pandas() if hasattr(cold_accounts, "to_pandas") else cold_accounts
print(f"Accounts with no contact in >30 days: {len(cold_accounts)}")
html_table(cold_accounts)